# 02 — Assign Gauge Stations to HUC12s

**Purpose:** Load WRTDS annual nitrogen load estimates and USGS gauge metadata,
filter to gauges with complete records in both analysis periods, spatially join
gauge locations to HUC12 polygons, and resolve HUC12s that contain multiple
gauges by retaining only the most downstream one.

**Inputs:**
- `UMRB_HUC12_modified.shp` (output of notebook 01)
- `load_cal_year_clean.csv` — annual nitrogen loads per gauge per year
- `gaugeSiteInfoUMRB.csv` — gauge metadata (lat/lon, drainage area)

**Output:** `cache/nb02_intermediate.pkl` (loaded by notebook 03)

**Gauge filtering rationale:**  
Only sites with records in **both** 2001–2005 and 2016–2020 are retained.
These windows define the early/late periods in the nitrogen trend decomposition
and bracket the major shift in conservation practice adoption across the UMRB.

In [ ]:
# =============================================================================
# USER SETTINGS — edit these paths before running
# =============================================================================

WORKING_DIR = "/path/to/your/data"

# Input: cleaned HUC12 shapefile from notebook 01
HUC12_CLEAN_SHP = f"{WORKING_DIR}/UMRB_HUC12_modified.shp"

# Input: annual nitrogen loads (WRTDS output; flux kg/day, annual load tonne/yr)
LOAD_CSV = f"{WORKING_DIR}/WRTDS/load_cal_year_clean.csv"

# Input: USGS gauge metadata
SITE_INFO_CSV = f"{WORKING_DIR}/gaugeSiteInfoUMRB.csv"

# Gauge year filter: retain only sites with records in ALL years of BOTH windows
# Early period = pre-2008 baseline; late period = recent decade
EARLY_YEARS = list(range(2001, 2006))   # 2001–2005
LATE_YEARS  = list(range(2016, 2021))   # 2016–2020

# Cache directory for NLDI API results and intermediate pipeline state
CACHE_DIR = "./cache"

# =============================================================================
import os
import sys
import pickle
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
%matplotlib inline

sys.path.insert(0, '.')
from utils.nldi_helpers import keep_largest_basin

os.makedirs(CACHE_DIR, exist_ok=True)

## 1. Load cleaned HUC12 shapefile

In [ ]:
UMRB_HUC12 = gpd.read_file(HUC12_CLEAN_SHP)
print(f'HUC12 polygons : {len(UMRB_HUC12)}')
print(f'CRS            : {UMRB_HUC12.crs}')

## 2. Load gauge data and build a point GeoDataFrame

In [ ]:
load = pd.read_csv(LOAD_CSV)

# Prepend 'USGS-' and zero-pad to 8 digits to match NLDI site identifier format
site_info = pd.read_csv(SITE_INFO_CSV)
site_info['site_no'] = 'USGS-' + site_info['site_no'].astype(str).str.zfill(8)

print(f'Load records   : {len(load):,} | Gauges in load file: {load["site_no"].nunique()}')
print(f'Gauge metadata : {len(site_info)} records')

In [ ]:
# CRS: EPSG:4269 (NAD83) — standard for USGS/NHD data
site_loc = (
    site_info
    .assign(geometry=lambda df: gpd.points_from_xy(df['dec_long_va'], df['dec_lat_va']))
    .pipe(gpd.GeoDataFrame, geometry='geometry', crs='EPSG:4269')
    .reset_index(drop=True)
)
print(f'Gauge point GeoDataFrame: {len(site_loc)} sites')

## 3. Filter gauges: require complete records in both analysis periods

In [ ]:
all_required_years = EARLY_YEARS + LATE_YEARS

filtered_sites = load.groupby('site_no').filter(
    lambda x: all(y in x['Year'].values for y in all_required_years)
)
filtered_site_loc = (
    site_loc[site_loc['site_no'].isin(filtered_sites['site_no'].unique())]
    .reset_index(drop=True)
)
print(f'Sites passing year filter: {len(filtered_site_loc)}')

## 4. Spatial join: assign each gauge to its containing HUC12

In [ ]:
# Left join retains all HUC12s; site_no is NaN where no gauge falls inside
UMRB_HUC12_with_sites = (
    gpd.sjoin(UMRB_HUC12, filtered_site_loc, how='left', predicate='contains')
    .drop(columns='index_right')
)

gauged = UMRB_HUC12_with_sites[UMRB_HUC12_with_sites['site_no'].notna()]
print(f'HUC12s with at least one gauge: {gauged["HUC_12"].nunique()}')

## 5. Resolve HUC12s with multiple gauges

When multiple gauges fall within the same HUC12, retain only the most downstream
one. It is identified as the gauge with the **largest upstream drainage basin**
fetched via the NLDI API — a larger basin means the gauge integrates flow from
further upstream and is therefore more downstream within the polygon.

In [ ]:
UMRB_HUC12_gauged = UMRB_HUC12_with_sites[UMRB_HUC12_with_sites['site_no'].notna()].copy()
grouped = UMRB_HUC12_gauged.groupby('HUC_12')['site_no'].unique()
multi_site = grouped[grouped.apply(len) > 1]

print(f'HUC12s with multiple gauges: {len(multi_site)}')
print(multi_site)

In [ ]:
# keep_largest_basin calls NLDI per site and caches results under CACHE_DIR/nldi/
nldi_cache = os.path.join(CACHE_DIR, 'nldi')
largest_sites = grouped.apply(lambda sites: keep_largest_basin(sites, cache_dir=nldi_cache))

# Report failures
failed = largest_sites[largest_sites.apply(lambda x: not isinstance(x, str))]
if not failed.empty:
    print(f'WARNING: could not determine downstream gauge for {len(failed)} HUC12(s):')
    print(failed)

largest_site_list = [s for s in largest_sites.tolist() if isinstance(s, str)]
print(f'Representative gauges retained: {len(largest_site_list)}')

In [ ]:
# Keep one gauge per HUC12
UMRB_HUC12_gauged = UMRB_HUC12_gauged[
    UMRB_HUC12_gauged['site_no'].isin(largest_site_list)
].copy()
print(f'Gauged HUC12 records after deduplication: {len(UMRB_HUC12_gauged)}')

## 6. Save intermediate state for notebook 03

In [ ]:
cache_pkl = os.path.join(CACHE_DIR, 'nb02_intermediate.pkl')
with open(cache_pkl, 'wb') as f:
    pickle.dump({
        'UMRB_HUC12'          : UMRB_HUC12,
        'UMRB_HUC12_with_sites': UMRB_HUC12_with_sites,
        'UMRB_HUC12_gauged'   : UMRB_HUC12_gauged,
        'filtered_site_loc'   : filtered_site_loc,
        'CACHE_DIR'           : CACHE_DIR,
    }, f)
print(f'Saved: {cache_pkl}')